In [ ]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install tensorflow


  Using cached tensorflow-2.19.0-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached absl_py-2.2.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-5.29.4-cp310-abi3-win_amd64.whl.metadata (592 bytes)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached termcolor-3.0.1-py3-none-any.whl.metadata (6.1 kB)
  Using cached wrapt-1.17.2-cp310-cp310-win_amd64.whl.metadata (6.5 kB)
  Using cached grpcio-1.71.0-cp310-cp310-win_amd64.whl.metadata (4.0 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cach

In [ ]:
!pip install opencv-python


In [ ]:
import cv2
print(cv2.__version__)


4.11.0


In [ ]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import cv2
from glob import glob
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.preprocessing import image
from matplotlib.image import imread
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.layers import BatchNormalization
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.models import Sequential, Model
from keras.regularizers import l2
from tensorflow.keras.layers import Activation, Dropout, Dense, Flatten, Conv2D, BatchNormalization, MaxPooling2D, GlobalAveragePooling2D,Input,concatenate, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import shutil
import random

# Define dataset paths
folder_path = "E:/cse366_research/OCT17/OCT2017"

train_dir = os.path.join(folder_path, "E:/cse366_research/OCT17/OCT2017/train")
val_dir = os.path.join(folder_path, "E:/cse366_research/OCT17/OCT2017/val")
test_dir = os.path.join(folder_path, "E:/cse366_research/OCT17/OCT2017t/test")



In [ ]:
pip install torch torchvision matplotlib tqdm


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision import models
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from tqdm import tqdm


In [ ]:

import torch
from torch.utils.data import Dataset
import os
from torchvision.io import read_image
import torchvision.transforms as T

class CustomImageDataset(Dataset):
    def __init__(self, ds_path, transforms=None, target_transform=None):
        self.ds_path = ds_path
        self.transforms = transforms
        self.target_transform = target_transform
        self.labels = os.listdir(ds_path)  # Get list of label folder names
        self.img_paths = []

        # Create a list of image paths and ensure all labels are mapped correctly
        for label in self.labels:
            base_path = os.path.join(ds_path, label)
            if os.path.isdir(base_path):  # Ensure it's a directory
                for img in os.listdir(base_path):
                    img_path = os.path.join(base_path, img)
                    self.img_paths.append(img_path)

        # Mapping labels to numeric values
        self.label_map = {

            'CNV': 0,

            'DME': 1,

            'DRUSEN': 2,

            'NORMAL': 3
        }

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = read_image(img_path)

        # Extract the label from the folder name in a cross-platform way
        label = os.path.basename(os.path.dirname(img_path))

        # Ensure label exists in label_map, otherwise raise an error
        if label not in self.label_map:
            raise KeyError(f"Label '{label}' not found in label_map. Check dataset structure.")

        image = image.float() / 255.0  # Normalize pixel values to [0,1]

        # Convert grayscale images to RGB if needed
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        # Apply transformations
        if self.transforms:
            image = self.transforms(image)

        image = T.Resize((224, 224))(image)  # Ensure image size is consistent

        return image, self.label_map[label]




In [ ]:
train_path = 'E:/cse366_research/OCT17/OCT2017/train'
train_data = CustomImageDataset(train_path)

val_path = 'E:/cse366_research/OCT17/OCT2017/val'
val_data = CustomImageDataset(val_path)

test_path = 'E:/cse366_research/OCT17/OCT2017/test'
test_data = CustomImageDataset(test_path)

train_len = len(train_data)
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)

val_len = len(val_data)
val_dataloader = DataLoader(val_data, batch_size=64, shuffle=True)

test_len = len(test_data)
test_dataloader = DataLoader(test_data, batch_size=4)

data_loaders = {'train': train_dataloader, 'val': val_dataloader}


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device


device(type='cuda', index=0)

In [ ]:
def __getitem__(self, idx):
    img_path = self.img_paths[idx]
    image = read_image(img_path)

    # Extract the label correctly for both Windows and Linux
    label = os.path.basename(os.path.dirname(img_path))  # Correct way to get folder name

    image = image.squeeze()
    image = image.repeat(3, 1, 1)
    image = image / 255.0
    if self.transforms:
        image = self.transforms(image)
    if self.target_transform:
        label = self.target_transform(label)

    image = T.Resize((224, 224))(image)

    return image, self.label_map[label]


In [ ]:
class ResNetLSTM(nn.Module):
    def __init__(self, num_classes=4, lstm_hidden_size=256, lstm_num_layers=1):
        super(ResNetLSTM, self).__init__()

        # Load pretrained ResNet18 model
        self.resnet = models.resnet18(pretrained=True)
        # Remove the final fully connected layer
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])

        # LSTM to process the features from ResNet
        self.lstm = nn.LSTM(input_size=512,  # The ResNet output size is 512
                            hidden_size=lstm_hidden_size,
                            num_layers=lstm_num_layers,
                            batch_first=True)

        # Fully connected layer for classification
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, x):
        # Pass through ResNet (without the final classification layer)
        x = self.resnet(x)  # Shape: [batch_size, 512, 1, 1]

        # Flatten ResNet output for LSTM
        x = x.view(x.size(0), -1)  # Shape: [batch_size, 512]
        x = x.unsqueeze(1)  # Adding sequence length dimension: [batch_size, 1, 512]

        # Pass through LSTM
        lstm_out, (hn, cn) = self.lstm(x)  # lstm_out: [batch_size, seq_len, hidden_size]

        # We only need the output from the last LSTM step
        lstm_out = lstm_out[:, -1, :]  # Shape: [batch_size, hidden_size]

        # Final classification
        out = self.fc(lstm_out)  # Shape: [batch_size, num_classes]
        return out


In [ ]:
model = ResNetLSTM(num_classes=4)
model.to(device)

# Criterion and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch


In [ ]:
def train_model(model, criterion, optimizer, data_loaders, device, num_epochs, patience=5):
    early_stop = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss, running_corrects = 0.0, 0
            total = 0

            for inputs, labels in tqdm(data_loaders[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                total += inputs.size(0)

            epoch_loss = running_loss / total
            epoch_acc = running_corrects.double() / total

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                early_stop.on_epoch_end(epoch, epoch_loss)
                if early_stop.early_stop:
                    print("Early stopping triggered.")
                    return history

    return history


In [ ]:
train_data = CustomImageDataset('E:/cse366_research/OCT17/OCT2017/train')
history=train_model(model, criterion, optimizer, data_loaders, device, num_epochs=5)



Epoch 1/5
----------


100%|██████████| 1305/1305 [05:34<00:00,  3.90it/s]


train Loss: 0.1409 Acc: 0.9523


100%|██████████| 1/1 [00:00<00:00,  3.69it/s]


val Loss: 0.0498 Acc: 1.0000
Epoch 2/5
----------


...

val Loss: 0.1772 Acc: 0.9375
Epoch 5/5
----------


100%|██████████| 1305/1305 [05:45<00:00,  3.78it/s]


train Loss: 0.0732 Acc: 0.9745


100%|██████████| 1/1 [00:00<00:00, 13.69it/s]

val Loss: 0.0383 Acc: 1.0000


In [ ]:
def evaluate_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    return accuracy


In [ ]:
test_acc = evaluate_accuracy(model, test_dataloader, device)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


Test Accuracy: 99.69%
